In [2]:
import torchvision
from kornia import filters as kf
from PIL import Image
import os
import torch
from torchvision.transforms.functional import to_pil_image
import random


def load_image(image_path):
    image = Image.open(image_path).convert('L')
    image_tensor = torchvision.transforms.ToTensor()(image)
    return image_tensor

def save_image(image_tensor, destination_dir):
    if len(image_tensor.shape) == 4:
        image_tensor = image_tensor.squeeze(0)
    pil_image = to_pil_image(image_tensor)
    image_path = destination_dir
    pil_image.save(image_path)

def apply_gaussian_blur(image_tensor, kernel_size=5, sigma=1.5):
    if len(image_tensor.shape) == 3:
        image_tensor = image_tensor.unsqueeze(0)
    image_blurred_tensor = kf.gaussian_blur2d(image_tensor, kernel_size, (sigma, sigma), border_type='constant')
    return image_blurred_tensor

def apply_motion_blur(image_tensor, kernel_size=11, angle=None, direction=0.0):
    if len(image_tensor.shape) == 3:
        image_tensor = image_tensor.unsqueeze(0)
    if angle is None:
        angle = random.uniform(-90, 90)
    image_blurred_tensor = kf.motion_blur(image_tensor, kernel_size, angle, direction, border_type='constant')
    return image_blurred_tensor

def apply_unsharp_mask(image_tensor, kernel_size=3, sigma=1.5):
    if len(image_tensor.shape) == 3:
        image_tensor = image_tensor.unsqueeze(0)
    image_blurred_tensor = kf.unsharp_mask(image_tensor, kernel_size, (sigma, sigma), border_type='constant')
    return image_blurred_tensor

def apply_salt_and_pepper_noise(image_tensor, noise_level=0.02, salt_vs_pepper=0.5):
    if len(image_tensor.shape) == 4:
        image_tensor = image_tensor.squeeze(0)
    if not (0 <= noise_level <= 1):
        raise ValueError("noise_level must be between 0 and 1")
    if not (0 <= salt_vs_pepper <= 1):
        raise ValueError("salt_vs_pepper must be between 0 and 1")
    noisy_image = image_tensor.clone()
    for c in range(image_tensor.shape[0]):
        noise = torch.rand(image_tensor.shape[1], image_tensor.shape[2])
        salt_mask = noise >= (1 - noise_level * salt_vs_pepper)
        noisy_image[c][salt_mask] = 1
        pepper_mask = noise <= (noise_level * (1 - salt_vs_pepper))
        noisy_image[c][pepper_mask] = 0
    if len(noisy_image.shape) == 4:
        return noisy_image
    elif len(noisy_image.shape) == 3:
        return noisy_image.unsqueeze(0)

def apply_filter_pipeline(image_tensor):
    image_tensor = apply_gaussian_blur(image_tensor)
    image_tensor = apply_motion_blur(image_tensor)
    image_tensor = apply_gaussian_blur(image_tensor)
    image_tensor = apply_salt_and_pepper_noise(image_tensor)
    return image_tensor


if __name__ == '__main__':
    source_dir = r'C:\Users\stefan.jovanovic_omi\Desktop\MasterElfak-NLP\b'
    destination_dir = r'C:\Users\stefan.jovanovic_omi\Desktop\MasterElfak-NLP\blur'
    os.makedirs(destination_dir, exist_ok=True)
    num_images = len(os.listdir(source_dir))

    for i, image_name in enumerate(os.listdir(source_dir)):
        # Load image
        image_path = os.path.join(source_dir, image_name)
        image = load_image(image_path)
        
        # Apply filter pipeline
        modified_image = apply_filter_pipeline(image)
        
        # Save image to new folder
        save_path = os.path.join(destination_dir, image_name)
        save_image(modified_image, save_path)
    